This notebook fits orthos to all replicates of the shendure-calibrated simulations

Imports

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-11-11 11:09:36.046464: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 11:09:36.049706: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

Autoreload for dev

In [2]:
%load_ext autoreload
%autoreload 2

Create a nice large cluster. We will need the resorces.

In [3]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=5,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=4:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=5)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Load the simulation object

In [4]:
#DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
DATA_ROOT="/home/mcn26/project_pi_skr2/shared/tabula_data"
simu_obj=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251111")

In [5]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj.orthos=[]

Fit orthos to all replicates

In [6]:
client.dashboard_link

'http://10.18.22.66:8787/status'

In [7]:
simu_obj.create_orthos_for_all_replicates(client)

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


Save

In [8]:
simu_obj.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_with_orthos_20251111")

In [ ]:
simu_obj.orthos[0].save(".","BLUH.ortho")

In [ ]:
simu_obj.orthos[0].by_cell_type_parameters

In [ ]:
simu_obj.orthos[0].by_cre_parameters

Shut down the cluster

In [10]:
client.close()
cluster.close()